# Module 2 — Black-76

Module 1 built the VIX futures curve. This module turns curve levels into **option prices**.

## The one-sentence version

**Black-76 is Black-Scholes with the spot price replaced by a forward price.**

That is genuinely almost the whole story. Same normal-distribution machinery, same
structure, one substitution. Fischer Black published it in 1976 specifically for options
on *futures*.

You already know Black-Scholes and implied vol, so this module spends its time on the
part that is specific to VIX: **why the forward, and what goes wrong if you use spot.**

No formulas below. Numbers and words only.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "learning" else Path.cwd()
sys.path.insert(0, str(REPO))
pd.set_option("display.width", 200)

from vixshock.pricing import black76, implied_vol

print("using the model's own pricing code from vixshock/pricing.py")

using the model's own pricing code from vixshock/pricing.py


## Step 1 — why the forward and not spot VIX

Two facts, and the second one does the work.

**Fact 1.** A VIX option and the VIX future of the *same expiry* settle to the same
number — spot VIX on that date.

**Fact 2.** Spot VIX is **not tradeable.** You cannot buy it. There is no spot VIX
position you can hold. So if you are short a VIX call and want to hedge, the only thing
you can actually trade is *the future*.

Pricing models are built on hedging arguments: an option is worth what it costs to
replicate. If you cannot trade spot VIX, spot VIX cannot be the underlying in your
pricing model. The future can.

This is why the model prices off the curve you built in Module 1, and why the code's
docstring says *"the forward is that future, not spot VIX."*

## Step 2 — what it costs to get this wrong

Same option, same strike, same vol, same expiry. Only the underlying changes.

In [2]:
# real values from the model's latest run
spot_vix = 17.71
curve = {35: 18.561, 63: 19.146, 91: 19.370, 126: 20.013}   # CM futures curve
vols  = {35: 0.925,  63: 0.773,  91: 0.725,  126: 0.674}    # VIX-option implied vols
RATE = 0.04
STRIKE = 20.0

rows = []
for dte, fwd in curve.items():
    yrs = dte / 365
    right = float(black76(fwd,      STRIKE, yrs, vols[dte], RATE, True))
    wrong = float(black76(spot_vix, STRIKE, yrs, vols[dte], RATE, True))
    rows.append({
        "days to expiry": dte,
        "future (correct)": fwd,
        "spot VIX (wrong)": spot_vix,
        "price off future": round(right, 3),
        "price off spot": round(wrong, 3),
        "underpriced by": round(right - wrong, 3),
        "error %": f"{(right-wrong)/wrong*100:.0f}%",
    })

print(f"A {STRIKE:.0f}-strike VIX CALL, priced two ways:\n")
print(pd.DataFrame(rows).to_string(index=False))
print("\nUsing spot instead of the future UNDERPRICES every one of these calls.")
print("The error grows with tenor, because the curve slopes upward away from spot.")

A 20-strike VIX CALL, priced two ways:

 days to expiry  future (correct)  spot VIX (wrong)  price off future  price off spot  underpriced by error %
             35            18.561             17.71             1.544           1.187           0.357     30%
             63            19.146             17.71             2.078           1.419           0.659     46%
             91            19.370             17.71             2.498           1.696           0.803     47%
            126            20.013             17.71             3.103           1.924           1.179     61%

Using spot instead of the future UNDERPRICES every one of these calls.
The error grows with tenor, because the curve slopes upward away from spot.


## Step 3 — moneyness: the thing you already worked out

You said earlier that the future's level determines the moneyness of an option expiring
at that date. That is exactly right, and here is why it matters *mechanically*.

Spot VIX is a single number. It has no tenor. So if you measure moneyness against spot,
**every option at the same strike looks identical**, regardless of expiry. Against the
correct forward, they are different options.

In [3]:
rows = []
for dte, fwd in curve.items():
    rows.append({
        "days to expiry": dte,
        "future": fwd,
        "future / strike": round(fwd / STRIKE, 4),
        "how it looks": ("AT the money" if abs(fwd/STRIKE - 1) < 0.01
                         else f"{(1-fwd/STRIKE)*100:.0f}% out of the money"),
        "spot / strike": round(spot_vix / STRIKE, 4),
    })

print(f"Moneyness of the {STRIKE:.0f}-strike call, measured correctly vs incorrectly:\n")
print(pd.DataFrame(rows).to_string(index=False))
print("\nLook at the last column: identical for all four. Spot cannot tell them apart.")
print("Look at 'how it looks': the 35-day option is well out of the money,")
print("while the 126-day option is essentially AT the money. Completely different risks.")

Moneyness of the 20-strike call, measured correctly vs incorrectly:

 days to expiry  future  future / strike        how it looks  spot / strike
             35  18.561           0.9281 7% out of the money         0.8855
             63  19.146           0.9573 4% out of the money         0.8855
             91  19.370           0.9685 3% out of the money         0.8855
            126  20.013           1.0007        AT the money         0.8855

Look at the last column: identical for all four. Spot cannot tell them apart.
Look at 'how it looks': the 35-day option is well out of the money,
while the 126-day option is essentially AT the money. Completely different risks.


### Why this drives the model's whole architecture

This is the payoff of Step 3, and it is worth holding on to.

Because each option's moneyness is set by the future **at its own expiry**, you cannot
shock VIX with a single number. A book with options at 35, 63, 91 and 126 days needs
**four different shocked underlyings.**

That is why the model shocks a *curve* rather than a level — and it is why the response
function has a tenor-damping dial at all. If moneyness were set by spot, one shock number
would do and half of this model would not need to exist.

## Step 4 — the three things that move an option's price here

Black-76 takes five inputs. Two are boring (the strike is fixed by the contract; the
interest rate barely matters at these maturities). Three do the work:

| input | in this model | where it comes from |
|---|---|---|
| **forward** | the VIX future for that expiry | the CM curve (Module 1) |
| **implied vol** | vol *of VIX options* — around 95% | the vol-of-vol data (Module 6) |
| **time** | days to expiry / 365 | the book |

A note on that implied vol, because it surprises people: **~95% is the volatility of
VIX itself**, not of SPX. VIX is a wildly more volatile thing than the stock market — it
can double in a week. So vols near 100% are normal here, not a data error. That is
exactly what the VVIX index measures.

The shock moves **two** of these three: the forward *and* the implied vol. Let's see
how much each one matters.

In [4]:
DTE = 35
F, V, yrs = curve[DTE], vols[DTE], DTE / 365
base = float(black76(F, STRIKE, yrs, V, RATE, True))

print(f"Baseline: {DTE}-day {STRIKE:.0f}-strike call, future {F}, vol {V*100:.1f}%")
print(f"  price = {base:.4f}\n")
print("Now change ONE input at a time:\n")

rows = []
for label, f2, v2 in [
    ("future +1 point",        F + 1, V),
    ("future +5 points",       F + 5, V),
    ("future +16 pts (-10% SPX shock)", F + 16.4, V),
    ("vol +10 points",         F, V + 0.10),
    ("vol +63 pts (-10% SPX shock)",   F, V + 0.628),
    ("BOTH (the real -10% shock)",     F + 16.4, V + 0.628),
]:
    p = float(black76(f2, STRIKE, yrs, v2, RATE, True))
    rows.append({"change": label, "future": round(f2, 2), "vol %": round(v2*100, 1),
                 "price": round(p, 4), "change in price": round(p - base, 4),
                 "x baseline": round(p / base, 2)})

print(pd.DataFrame(rows).to_string(index=False))
print("\nBoth channels matter. The forward move dominates, but the vol move is not")
print("a rounding error - which is the entire reason the model has a vol-of-vol layer.")

Baseline: 35-day 20-strike call, future 18.561, vol 92.5%
  price = 1.5439

Now change ONE input at a time:

                         change  future  vol %   price  change in price  x baseline
                future +1 point   19.56   92.5  2.0321           0.4881        1.32
               future +5 points   23.56   92.5  4.6345           3.0906        3.00
future +16 pts (-10% SPX shock)   34.96   92.5 14.9763          13.4324        9.70
                 vol +10 points   18.56  102.5  1.7713           0.2273        1.15
   vol +63 pts (-10% SPX shock)   18.56  155.3  2.9760           1.4320        1.93
     BOTH (the real -10% shock)   34.96  155.3 15.6580          14.1140       10.14

Both channels matter. The forward move dominates, but the vol move is not
a rounding error - which is the entire reason the model has a vol-of-vol layer.


## Step 5 — one real position, all the way through

This is the full chain the model runs for every option in the book, on one position from
the example book: **long 100 of the Oct-21 20-strike call.**

Five steps: find the forward, find the vol, price it, shock both inputs, re-price.

In [4]:
from vixshock.response import load_params
from vixshock.volofvol import load_vov_params

vix_params = load_params()
vov_params = load_vov_params()

DTE, K, QTY, SHOCK = 35, 20.0, 100, -0.10
yrs = DTE / 365

print(f"POSITION: long {QTY} x  {DTE}-day  {K:.0f}-strike VIX CALL")
print(f"SCENARIO: SPX falls {abs(SHOCK)*100:.0f}%\n")
print("-" * 66)

F0 = curve[DTE]
V0 = vols[DTE]
print(f"1. forward  = VIX future for this expiry        = {F0:.3f}")
print(f"2. vol      = VIX-option implied vol at 35d     = {V0*100:.1f}%")

p0 = float(black76(F0, K, yrs, V0, RATE, True))
print(f"3. price    = Black-76(...)                     = {p0:.4f}")
print(f"   position value = {QTY} x {p0:.4f}                 = {QTY*p0:.2f}")
print("-" * 66)

dF = float(vix_params.dvix(SHOCK, DTE))
dV = float(vov_params.dvix(SHOCK, DTE)) / 100
print(f"4. the shock, read at THIS option's tenor ({DTE} days):")
print(f"     forward moves  {dF:+.3f} points   -> {F0:.3f} -> {F0+dF:.3f}")
print(f"     vol moves      {dV*100:+.1f} points   -> {V0*100:.1f}% -> {(V0+dV)*100:.1f}%")
print("-" * 66)

p_fixed = float(black76(F0 + dF, K, yrs, V0,      RATE, True))
p_full  = float(black76(F0 + dF, K, yrs, V0 + dV, RATE, True))
print(f"5. re-price:")
print(f"     vol held fixed  = {p_fixed:.4f}   P&L = {QTY*(p_fixed-p0):+.2f}")
print(f"     vol shocked too = {p_full:.4f}   P&L = {QTY*(p_full-p0):+.2f}")
print(f"\n   the vol-of-vol layer added {QTY*(p_full-p_fixed):+.2f} to this position's P&L")

POSITION: long 100 x  35-day  20-strike VIX CALL
SCENARIO: SPX falls 10%

------------------------------------------------------------------
1. forward  = VIX future for this expiry        = 18.561
2. vol      = VIX-option implied vol at 35d     = 92.5%
3. price    = Black-76(...)                     = 1.5439
   position value = 100 x 1.5439                 = 154.39
------------------------------------------------------------------
4. the shock, read at THIS option's tenor (35 days):
     forward moves  +15.869 points   -> 18.561 -> 34.430
     vol moves      +60.9 points   -> 92.5% -> 153.4%
------------------------------------------------------------------
5. re-price:
     vol held fixed  = 14.4572   P&L = +1291.33
     vol shocked too = 15.1412   P&L = +1359.73

   the vol-of-vol layer added +68.40 to this position's P&L


## Step 6 — the same option at four different expiries

Now the point of Step 3, made concrete. Same strike, same shock — but each expiry reads
the shock at *its own* tenor, so each gets a different forward move.

In [5]:
rows = []
for dte in curve:
    yrs_i, F0_i, V0_i = dte/365, curve[dte], vols[dte]
    dF_i = float(vix_params.dvix(SHOCK, dte))
    dV_i = float(vov_params.dvix(SHOCK, dte))/100
    p0_i = float(black76(F0_i, K, yrs_i, V0_i, RATE, True))
    p1_i = float(black76(F0_i+dF_i, K, yrs_i, V0_i+dV_i, RATE, True))
    rows.append({"days": dte, "future": F0_i, "fwd moves": round(dF_i,2),
                 "shocked fwd": round(F0_i+dF_i,2), "vol moves": round(dV_i*100,1),
                 "price before": round(p0_i,3), "price after": round(p1_i,3),
                 "x": round(p1_i/p0_i,1)})

print(f"{K:.0f}-strike call at four expiries, SPX {SHOCK*100:.0f}%:\n")
print(pd.DataFrame(rows).to_string(index=False))
print("\nThe 35-day forward moves much more than the 126-day forward.")
print("Same shock, different response by tenor. THAT is what the model's")
print("tenor-damping dial encodes - and why one shock number would not do.")

20-strike call at four expiries, SPX -10%:

 days  future  fwd moves  shocked fwd  vol moves  price before  price after   x
   35  18.561      15.87        34.43       60.9         1.544       15.141 9.8
   63  19.146      13.16        32.31       51.4         2.078       13.556 6.5
   91  19.370      10.91        30.28       43.4         2.498       12.093 4.8
  126  20.013       8.64        28.65       35.1         3.103       10.885 3.5

The 35-day forward moves much more than the 126-day forward.
Same shock, different response by tenor. THAT is what the model's
tenor-damping dial encodes - and why one shock number would not do.


## Step 7 — implied vol, round-tripped

You described implied vol as "backing volatility out of the market price through
Black-Scholes." The model has a function that does exactly that (`implied_vol` in
`vixshock/pricing.py`), and it is a good sanity check on the pricer: price an option at
a known vol, then ask the code to recover that vol from the price. You should get back
what you put in.

In [6]:
F, K2, yrs2 = 18.561, 20.0, 35/365
print("price at a known vol, then recover the vol from the price:\n")
for v_in in (0.60, 0.925, 1.20):
    px = float(black76(F, K2, yrs2, v_in, RATE, True))
    v_out = implied_vol(px, F, K2, yrs2, RATE, True)
    print(f"  vol in {v_in*100:6.1f}%  ->  price {px:7.4f}  ->  vol out {v_out*100:6.1f}%")
print("\nRound-trips cleanly, so the pricer and the inverter agree.")

price at a known vol, then recover the vol from the price:

  vol in   60.0%  ->  price  0.8177  ->  vol out   60.0%
  vol in   92.5%  ->  price  1.5439  ->  vol out   92.5%
  vol in  120.0%  ->  price  2.1705  ->  vol out  120.0%

Round-trips cleanly, so the pricer and the inverter agree.


---

## What to take away

**What Black-76 is:** Black-Scholes with a forward price in place of spot. Published 1976
for options on futures.

**Why it is the right choice here:** VIX options settle to the same number as the VIX
future of matching expiry, and spot VIX is not tradeable, so the future is both the
hedge instrument and the pricing underlying. Using spot underprices these calls badly,
and the error grows with tenor.

**Why moneyness matters structurally:** each option's moneyness is set by the future at
*its own* expiry. Spot has no tenor, so it cannot distinguish a 35-day option from a
126-day one. This is why the model shocks a whole curve and damps by tenor.

**The ~95% implied vols are correct, not a bug:** that is the volatility of VIX itself,
which is far more volatile than SPX.

**Both channels are shocked:** forward *and* implied vol. The model always reports P&L
both ways so the vol contribution is never hidden.

**Honest limitations, for the validation role:**

1. **Black-76 assumes the forward is lognormal.** VIX is mean-reverting, not lognormal —
   it does not wander off to 500. So this is a *quoting convention*, not a belief about
   VIX dynamics. Defensible because the market itself quotes VIX options this way, so the
   vols being fed in were derived under the same convention. It is internally consistent.
2. **One vol per tenor — no skew.** Real VIX options have a pronounced skew (upside calls
   trade at much higher vols). The model uses a single at-the-money vol per tenor, so a
   book concentrated in far out-of-the-money calls will be mispriced at the base level.
   Worth flagging if your real book is skew-heavy.
3. **Time does not pass.** The shock is instantaneous unless you pass `--days-forward`.
   No theta.

**Code:** `vixshock/pricing.py` is 42 lines — the whole pricer. `vixshock/shock.py` runs
the chain from Step 5 over every option in the book.

---

### Questions to test yourself

1. Why not Black-Scholes with spot VIX? Give the hedging argument, not just "it settles
   to the future."
2. A colleague sees implied vol of 95% and says the data is broken. What do you say?
3. Why can't a single shocked VIX number reprice a book of options at different expiries?
4. Someone says "Black-76 assumes lognormal, but VIX mean-reverts, so this is wrong."
   What is the defense?

Next: **Module 3 — the response function**, the heart of the model: the three dials that
turn an SPX move into a curve move.

---

## Answer key — the two arguments worth memorising

The other two questions have short answers. These two do not, and they are the ones a
quant reviewer actually asks.

### Q1. Why not Black-Scholes with spot VIX? (the hedging argument)

Every option pricing model works by **replication**: *I can build a portfolio that
behaves like this option, so the option must cost what that portfolio costs, or there is
free money.* Black-Scholes replicates a stock option by continuously holding some amount
of the stock — delta hedging.

So the real question is not "what does the option settle against?" but **"what can I
actually trade to hedge it?"**

You cannot buy spot VIX. It is not an asset — it is a *calculation*, a number CBOE
publishes from a strip of SPX option prices. There is no instrument, no position, nothing
to hold. A model saying "hedge this VIX call with delta units of spot VIX" describes an
operation that cannot be performed. The replication argument does not become inaccurate;
it becomes **meaningless**.

The VIX *future* is tradeable. Short a VIX call, you hedge with VIX futures — and at
expiry the future and the option settle to the same number, so the hedge converges.

> **Say this:** "Pricing models price by replication, and you can only replicate with
> things you can trade. Spot VIX isn't an instrument, it's an index calculation — there
> is nothing to hold. The future is what a desk actually hedges with, so the future is
> the underlying. Black-76 is just Black-Scholes rewritten to take a forward."

Two supporting points if pressed:

- **No cost-of-carry links spot VIX to the future.** With gold, spot x financing =
  forward, so spot and forward are interchangeable inputs. With VIX that link does not
  exist — you cannot store VIX. You cannot back into the future from spot even in
  principle.
- This is why the term premium (+1.52 calm, -3.16 stressed) persists for decades.
  Nobody can arbitrage it flat.

### Q4. "Black-76 assumes lognormal but VIX mean-reverts, so this is wrong."

Careful — this is **not** the same question as Q1. Answering "because the underlying is a
future" does not address it: the reviewer already knows that, and the objection is about
the *distribution*, not the instrument. Stated fully, the challenge is:

> *"Black-76 assumes the underlying is lognormal — it can drift anywhere and never
> returns. But VIX mean-reverts: it does not wander to 500, it gets pulled back to ~20.
> Your distributional assumption is false, so your prices are wrong."*

Three layers of defense:

**1. It is a quoting convention, not a belief.** Black-76 here is a translator between
price and vol, not a claim about how VIX behaves. Nobody at CBOE thinks VIX is lognormal.

**2. It is internally consistent — this is the real defense.** The implied vols fed into
this model (VVIX, Bloomberg ATM vols) were *themselves* produced by inverting market
prices through Black-76. So the model goes market price -> vol -> price under the **same**
convention in both directions. **The convention cancels.** You would get the same answer
with any invertible formula, as long as you use the same one both ways. Step 7 above
demonstrates the round-trip.

**3. Where it would actually bite.** The convention stops cancelling only when you
extrapolate *away* from where the vol was observed — pricing a far out-of-the-money
strike using an at-the-money vol. That is the **skew** limitation (limitation 2 above),
and it is the genuine weakness. Note it is a different criticism from the one asked,
which is exactly why conceding it makes you credible.

> **Say this:** "It's a quoting convention, not a distributional belief. The vols we feed
> in were backed out of market prices using Black-76, so we go price to vol to price
> under the same convention and it cancels. Where it *would* bite is extrapolating across
> strikes — and that is a real limitation: we use one ATM vol per tenor and don't model
> skew."

**Then do this:** check whether the real book has meaningful out-of-the-money exposure.
If it does, that is the limitation that actually costs money — and you will have found it
before anyone asked.

### Q3 — one addition

Mean reversion anchoring the long end is the right *economic* reason. The *mechanical*
reason is more fundamental: each option's moneyness is set by the future at its own
expiry, and those are different instruments at different prices. Even with no mean
reversion at all, one number could not reprice them.

Also know that this model **approximates** mean reversion rather than modelling it. The
tenor damping is multiplicative — the long end moves a fixed *fraction* of the front.
True mean reversion would be an **anchor**: far-dated futures pulled toward a long-run
level regardless of starting point. README limitation 1 concedes this. The evidence it is
good enough: the 120-day fit is as good as the 30-day fit. But if someone says "that is
not really mean reversion," they are right — it is a damping approximation that holds
over the observed range, not an anchor term.